# Week 6, Lab 4 — Two MCP servers, one agent


In [ ]:
WEEK = 'Week 6'
LAB = 'Lab 4 — multiple servers'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVERS = {
    "tools": StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "local_tools_server.py")]),
    "notes": StdioServerParameters(command="python", args=[str(ROOT / "6_mcp" / "servers" / "notes_server.py")]),
}

async def call(server: str, tool: str, args: dict):
    async with stdio_client(SERVERS[server]) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            return await s.call_tool(tool, args)

print("calc", await call("tools", "calculator", {"expression": "2+2"}))
print("note", await call("notes", "add_note", {"text": "MCP servers are just processes."}))
print("list", await call("notes", "list_notes", {}))


Write a ReAct loop that picks (server, tool) from a merged catalog.
